# JEPA House — Entraînement

Encodeur statistiques LLM + encodeur musical house alignés dans un espace latent commun.
Architecture BYOL (deux encodeurs asymétriques, EMA sur le target).

## Structure du notebook

| Cellule | Contenu |
|---------|----------|
| 1 | Install + imports |
| 2 | Config hyperparams |
| 3 | Pipeline données — Lakh MIDI (filtre house) |
| 4 | Corpus LLM — features statistiques depuis les notebooks poids |
| 5 | Architecture JEPA (Enc_stats, Enc_music, Predictor, EMA) |
| 6 | Training loop (3 phases) |
| 7 | Évaluation — alignement + métriques musicales |
| 8 | Export des artefacts |
| 9 | Intégration — génération avec JEPA (remplacement du LSTM) |

## Ce que le JEPA apprend

Aligner deux espaces hétérogènes :

- **Enc_stats** : `SV[q,k,v,gate,down] × NL` + `alpha/bimod × NL` + `Wasserstein[NL]` + `qk_coh[NL]` → `z_stats [D_lat=256]`
- **Enc_music** : fenêtres de 32 notes house (pitch, dur_beats, vel, interval, beat_pos) → `z_music [D_lat=256]`
- **Predictor** : `z_stats → z_stats_pred` dans l'espace de `z_music`
- **Loss** : `1 - cosine(z_stats_pred, sg(z_music_target))` + variance régularisation + continuité temporelle

## Spécificité house

Le filtre Lakh retient uniquement les fichiers 115–142 BPM, 4/4, ≥32 bars, ≥8 notes/bar.
La `beat_pos` (position 16th dans le bar) est une feature centrale — elle encode le groove house (kick sur tous les beats, bassline syncopée).
L'appariement LLM↔MIDI est guidé par curriculum : `alpha < 1.3` (impulsif) → fenêtres à forte densité rythmique ; `alpha > 1.8` (gaussien) → fenêtres legato.

In [ ]:
!pip install -q pretty_midi tqdm einops datasets transformers accelerate scikit-learn
# Lakh MIDI Dataset téléchargement (nécessite ~3GB)
import os
if not os.path.exists("/content/lmd_matched"):
    print("Téléchargement Lakh MIDI Dataset...")
    !wget -q "http://hog.ee.columbia.edu/craffel/lmd/lmd_matched.tar.gz" -O /tmp/lmd.tar.gz
    !tar -xzf /tmp/lmd.tar.gz -C /content/
    print("  Lakh extrait")
else:
    print("Lakh déjà présent")
from google.drive import drive  # pour sauvegarder les checkpoints
import torch
print("PyTorch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "—", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm
import pretty_midi, json, os, time, copy, einops
import matplotlib.pyplot as plt

# ── Hyperparamètres ──────────────────────────────────────────────────────────
CFG = {
    # Architecture
    "D_LAT"        : 256,    # dimension espace latent commun
    "D_HIDDEN"     : 128,    # hidden dim Transformer
    "N_HEADS"      : 4,      # têtes attention
    "N_TRANS"      : 3,      # couches Transformer dans chaque encodeur
    "D_PRED"       : 512,    # hidden MLP Predictor
    "WINDOW"       : 32,     # notes par fenêtre MIDI
    "STRIDE"       : 8,      # stride fenêtrage
    "NOTE_FEAT"    : 5,      # pitch,dur,vel,interval,beat_pos
    # LLM
    "NL"           : 18,     # layers (Gemma-3-270M)
    "N_SV_PROJ"    : 5,      # q,k,v,gate,down
    "D_SV"         : 64,     # SVs par projection
    # Entraînement
    "BATCH"        : 128,
    "LR"           : 3e-4,
    "LR_FROZEN"    : 3e-5,   # phase 3 fine-tuning
    "WD"           : 0.01,
    "TAU_INIT"     : 0.996,
    "TAU_FINAL"    : 0.9999,
    "DROPOUT"      : 0.1,
    "GRAD_CLIP"    : 1.0,
    "EPOCHS_P1"    : 50,     # phase 1 : musique seule
    "EPOCHS_P2"    : 50,     # phase 2 : alignement (Enc_music gelé)
    "EPOCHS_P3"    : 30,     # phase 3 : fine-tuning joint
    "WARMUP"       : 5,
    # Données house
    "BPM_MIN"      : 115,
    "BPM_MAX"      : 142,
    "MIN_BARS"     : 32,
    "MIN_NOTE_DENSITY": 8,   # notes/bar minimum
    # Loss
    "LAMBDA_VAR"   : 0.01,
    "LAMBDA_TEMP"  : 0.10,
    "GAMMA_VAR"    : 1.0,    # variance minimale cible
    "MARKOV_W"     : 0.70,   # poids Markov vs JEPA à l'inférence
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)

def norm01_np(x):
    mn,mx=x.min(),x.max(); return (x-mn)/(mx-mn+1e-9)

print("Config OK — device:", DEVICE)
print("D_LAT=%d  WINDOW=%d  BATCH=%d" % (CFG["D_LAT"],CFG["WINDOW"],CFG["BATCH"]))


In [ ]:
# ══════════════════════════════════════════════════════════
# PIPELINE DONNÉES — Lakh MIDI Dataset (filtre house)
# ══════════════════════════════════════════════════════════

def is_house_file(midi_path, cfg):
    """
    Retourne (True, bpm, piste_principale) si le fichier est house-compatible.
    Critères : BPM 115-142, 4/4, ≥32 bars, ≥8 notes/bar.
    """
    try:
        pm = pretty_midi.PrettyMIDI(str(midi_path))
    except:
        return False, None, None

    # BPM
    tempos = pm.get_tempo_changes()[1]
    if len(tempos) == 0: return False, None, None
    bpm = float(np.median(tempos))
    if not (cfg["BPM_MIN"] <= bpm <= cfg["BPM_MAX"]):
        return False, None, None

    # Time signature
    ts = pm.time_signature_changes
    if ts and ts[0].numerator != 4:
        return False, None, None

    # Piste principale : la plus longue en nombre de notes
    if not pm.instruments:
        return False, None, None
    inst = max(pm.instruments, key=lambda i: len(i.notes))
    if len(inst.notes) < 32: return False, None, None

    # Durée en bars
    beat_s = 60.0 / bpm
    bar_s  = beat_s * 4
    n_bars = pm.get_end_time() / bar_s
    if n_bars < cfg["MIN_BARS"]: return False, None, None

    # Densité
    density = len(inst.notes) / max(n_bars, 1)
    if density < cfg["MIN_NOTE_DENSITY"]: return False, None, None

    return True, bpm, inst


def extract_windows(instrument, bpm, window=32, stride=8):
    """
    Extrait des fenêtres de `window` notes depuis une piste MIDI.
    Features/note : [pitch_n, dur_n, vel_n, interval_n, beat_pos_n]
    Retourne : list de np.float32 [window, 5]
    """
    beat_s = 60.0 / bpm
    notes  = sorted(instrument.notes, key=lambda n: n.start)
    N = len(notes)
    if N < window: return []

    windows = []
    for start in range(0, N - window + 1, stride):
        chunk = notes[start:start+window]
        feat  = []
        for i, note in enumerate(chunk):
            dur_beats   = (note.end - note.start) / beat_s
            interval    = note.pitch - chunk[i-1].pitch if i > 0 else 0
            onset_beats = note.start / beat_s
            beat_pos    = int(onset_beats % 4.0 * 4) % 16  # 16th dans le bar
            feat.append([
                note.pitch / 127.0,
                float(np.log1p(dur_beats) / np.log1p(8.0)),
                note.velocity / 127.0,
                float(np.clip(interval / 24.0, -1.0, 1.0)),
                beat_pos / 15.0,
            ])
        windows.append(np.array(feat, dtype=np.float32))
    return windows


def compute_window_entropy(window_feat):
    """
    Entropie harmonique de la fenêtre = entropie de la distribution des pitches.
    Haute entropie → fenêtre chromatique/dissonante (haute tension).
    Basse entropie → fenêtre diatonique/stable.
    """
    pitches = (window_feat[:, 0] * 127).astype(int) % 12
    counts  = np.bincount(pitches, minlength=12).astype(float)
    p = counts / (counts.sum() + 1e-9)
    return float(-np.sum(p * np.log(p + 1e-12)))


print("Scan Lakh MIDI Dataset (filtre house)...")
LAKH_ROOT = Path("/content/lmd_matched")
midi_files = list(LAKH_ROOT.rglob("*.mid"))[:]
print(f"  {len(midi_files):,} fichiers trouvés")

all_windows     = []   # list de np.float32 [WINDOW, 5]
all_entropies   = []   # float par fenêtre
all_bpms        = []   # bpm par fenêtre
n_files_ok      = 0

for path in tqdm(midi_files, desc="Filtrage house"):
    ok, bpm, inst = is_house_file(path, CFG)
    if not ok: continue
    windows = extract_windows(inst, bpm,
                              window=CFG["WINDOW"],
                              stride=CFG["STRIDE"])
    if not windows: continue
    for w in windows:
        all_windows.append(w)
        all_entropies.append(compute_window_entropy(w))
        all_bpms.append(bpm)
    n_files_ok += 1

print(f"\n  {n_files_ok:,} fichiers house retenus")
print(f"  {len(all_windows):,} fenêtres extraites")

# Sauvegarder pour réutilisation
np.save("lakh_house_windows.npy",   np.stack(all_windows))
np.save("lakh_house_entropies.npy", np.array(all_entropies))
np.save("lakh_house_bpms.npy",      np.array(all_bpms))

# Distribution de l'entropie
plt.figure(figsize=(8,3))
plt.hist(all_entropies, bins=40, color="#4A90D9", alpha=0.82, edgecolor="white")
plt.axvline(np.percentile(all_entropies,33), color="green", ls="--", label="Q33 (stable)")
plt.axvline(np.percentile(all_entropies,67), color="red",   ls="--", label="Q67 (tendu)")
plt.title("Distribution entropie harmonique — corpus house")
plt.xlabel("Entropie"); plt.ylabel("Count"); plt.legend()
plt.tight_layout(); plt.savefig("logs/data_entropy_dist.png", dpi=120); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════
# CORPUS LLM — features statistiques
# Réutilise les SV calculés dans les notebooks poids
# ══════════════════════════════════════════════════════════

from transformers import AutoModelForCausalLM
from sklearn.decomposition import TruncatedSVD
import gc

MODEL_ID = "google/gemma-3-270m"

# ── Option A : recalculer depuis le modèle ───────────────────────────────────
# (utiliser si pas de fichier .npz pré-calculé)

def extract_llm_features(model_id, device="cpu"):
    """
    Extrait les features statistiques du modèle :
    SV[q,k,v,gate,down] × NL × 64, alpha × NL, bimod × NL, wdists × NL, qk_cohs × NL.
    Retourne un dict de np.float32.
    """
    from scipy.stats import kurtosis as sp_kurt, skew as sp_skew
    from scipy.signal import find_peaks
    from scipy.stats import wasserstein_distance

    print(f"  Chargement {model_id}...")
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
    model.eval()
    sd  = {k:v.detach().float().numpy() for k,v in model.state_dict().items()}
    NL_ = model.config.num_hidden_layers
    del model; gc.collect()

    def W(l, name):
        keys = [k for k in sd if f"layers.{l}." in k and name in k]
        return sd[keys[0]] if keys else None

    # SV par projection
    SV = {}
    for tn in ["q_proj","k_proj","v_proj","gate_proj","down_proj"]:
        rows = []
        for l in range(NL_):
            Wt = W(l, tn)
            if Wt is not None and Wt.ndim == 2:
                k = min(64, min(Wt.shape)-1)
                sv = TruncatedSVD(n_components=max(k,2),
                                  random_state=42).fit(Wt).singular_values_[:64] if k>=2 else np.ones(64)
            else: sv = np.ones(64)
            if len(sv) < 64: sv = np.pad(sv, (0, 64-len(sv)))
            rows.append(sv.astype(np.float32))
        SV[tn] = np.array(rows)  # [NL, 64]

    # Micro : alpha-stable, bimodalité
    alphas = []; bimodalities = []
    for l in range(NL_):
        Wt = W(l,"gate_proj") or W(l,"up_proj")
        if Wt is None:
            alphas.append(2.0); bimodalities.append(0.33); continue
        x = Wt.ravel()[:3000]
        kurt = float(sp_kurt(x)); skw = float(sp_skew(x))
        alpha = float(np.clip(2.0/(1.0+max(0,(kurt-3)/10.0)),0.5,2.0))
        n = len(x)
        sarle = (skw**2+1)/(kurt+3*(n-1)**2/(max(n-2,1)*max(n-3,1))+1e-9)
        alphas.append(alpha)
        bimodalities.append(float(np.clip(sarle,0,1)))

    # Macro : Wasserstein + QK cohérence
    wdists = np.zeros(NL_)
    for l in range(NL_-1):
        wdists[l] = wasserstein_distance(SV["q_proj"][l], SV["q_proj"][l+1])
    qk_cohs = np.zeros(NL_)
    for l in range(NL_):
        sq = np.abs(np.fft.rfft(SV["q_proj"][l],n=32))**2
        sk = np.abs(np.fft.rfft(SV["k_proj"][l],n=32))**2
        sq/=(sq.sum()+1e-12); sk/=(sk.sum()+1e-12)
        c = np.corrcoef(sq,sk)[0,1]
        qk_cohs[l] = float(c) if np.isfinite(c) else 0.0

    return {
        "sv_q":    SV["q_proj"].astype(np.float32),
        "sv_k":    SV["k_proj"].astype(np.float32),
        "sv_v":    SV["v_proj"].astype(np.float32),
        "sv_gate": SV["gate_proj"].astype(np.float32),
        "sv_down": SV["down_proj"].astype(np.float32),
        "alpha":   np.array(alphas, dtype=np.float32),
        "bimod":   np.array(bimodalities, dtype=np.float32),
        "wdists":  wdists.astype(np.float32),
        "qk_cohs": qk_cohs.astype(np.float32),
        "NL": NL_,
    }


# ── Option B : charger depuis un fichier pré-calculé ────────────────────────
LLM_FEAT_PATH = "llm_features.npz"

if os.path.exists(LLM_FEAT_PATH):
    print(f"Chargement features LLM depuis {LLM_FEAT_PATH}...")
    feat_dict = dict(np.load(LLM_FEAT_PATH))
    CFG["NL"] = int(feat_dict["NL"])
else:
    print("Calcul features LLM (peut prendre quelques minutes)...")
    feat_dict = extract_llm_features(MODEL_ID)
    np.savez(LLM_FEAT_PATH, **feat_dict)
    print(f"  Sauvegardé: {LLM_FEAT_PATH}")

NL = CFG["NL"]
print(f"Features LLM OK — NL={NL}")
print(f"  SV shape: {feat_dict["sv_q"].shape}")
print(f"  alpha range: [{feat_dict["alpha"].min():.2f}, {feat_dict["alpha"].max():.2f}]")
print(f"  bimod range: [{feat_dict["bimod"].min():.2f}, {feat_dict["bimod"].max():.2f}]")

# Tension globale = alpha_moyen (proxy : layers impulsifs → haute tension musicale)
# alpha faible (impulsif) → haute tension → fenêtres chromatiques
# alpha élevé (gaussien)  → faible tension → fenêtres diatoniques
LLM_TENSION = float(1.0 - (feat_dict["alpha"].mean() - 0.5) / 1.5)
print(f"  Tension LLM globale: {LLM_TENSION:.3f}  (0=calme, 1=tendu)")


In [ ]:
# ══════════════════════════════════════════════════════════
# ARCHITECTURE JEPA HOUSE
# Deux encodeurs asymétriques (online/target) + EMA
# ══════════════════════════════════════════════════════════

import torch, torch.nn as nn, torch.nn.functional as F
from einops import rearrange, reduce


# ── Encodeur statistiques LLM ────────────────────────────────────────────────
class SVProjector(nn.Module):
    """Linear(D_SV → 32) + LayerNorm par projection SV."""
    def __init__(self, d_sv=64, d_out=32):
        super().__init__()
        self.proj = nn.Linear(d_sv, d_out)
        self.norm = nn.LayerNorm(d_out)
    def forward(self, x):  # x: [B, NL, D_SV]
        return self.norm(self.proj(x))


class MicroEncoder(nn.Module):
    """MLP(3 → 16) pour alpha, bimod, tail_density."""
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2, 16), nn.GELU(), nn.Linear(16, 16))
    def forward(self, x):  # x: [B, NL, 2] (alpha, bimod)
        return self.mlp(x)


class StatsEncoder(nn.Module):
    """
    Enc_stats : SV×5 + micro + wdists + qk_cohs → z_stats [D_LAT]
    Input par layer : [5×32 + 16 + 1 + 1] = 178 dim
    Transformer 3 couches sur la séquence [NL, 178] → mean pool → Linear → [D_LAT]
    """
    def __init__(self, cfg):
        super().__init__()
        self.NL      = cfg["NL"]
        self.D_SV    = cfg["D_SV"]
        self.N_PROJ  = cfg["N_SV_PROJ"]  # 5
        D_LAT = cfg["D_LAT"]

        self.sv_projs   = nn.ModuleList([SVProjector(self.D_SV, 32)
                                         for _ in range(self.N_PROJ)])
        self.micro_enc  = MicroEncoder()
        # concat dim = 5×32 + 16 + 2 (wdist + qk_coh) = 178
        self.concat_dim = self.N_PROJ * 32 + 16 + 2

        # Projection vers D_HIDDEN avant Transformer
        self.input_proj = nn.Linear(self.concat_dim, cfg["D_HIDDEN"])
        self.pos_emb    = nn.Embedding(self.NL, cfg["D_HIDDEN"])

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg["D_HIDDEN"], nhead=cfg["N_HEADS"],
            dim_feedforward=cfg["D_HIDDEN"]*4,
            dropout=cfg["DROPOUT"], batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer,
                                                  num_layers=cfg["N_TRANS"])
        self.pool_norm  = nn.LayerNorm(cfg["D_HIDDEN"])
        self.out_proj   = nn.Linear(cfg["D_HIDDEN"], D_LAT)

    def forward(self, sv_tensors, alpha, bimod, wdists, qk_cohs):
        """
        sv_tensors : list de 5 tenseurs [B, NL, D_SV]
        alpha, bimod, wdists, qk_cohs : [B, NL]
        Retourne : z_stats [B, D_LAT]
        """
        B = sv_tensors[0].shape[0]
        # SV encodés
        sv_enc = [proj(sv) for proj, sv in zip(self.sv_projs, sv_tensors)]  # 5×[B,NL,32]
        sv_cat = torch.cat(sv_enc, dim=-1)  # [B, NL, 160]
        # Micro encodé
        micro_in = torch.stack([alpha, bimod], dim=-1)  # [B, NL, 2]
        micro_enc = self.micro_enc(micro_in)             # [B, NL, 16]
        # Concat tout
        x = torch.cat([
            sv_cat, micro_enc,
            wdists.unsqueeze(-1),   # [B, NL, 1]
            qk_cohs.unsqueeze(-1),  # [B, NL, 1]
        ], dim=-1)  # [B, NL, 178]
        # Transformer
        pos = self.pos_emb(torch.arange(self.NL, device=x.device))  # [NL, D_H]
        x = self.input_proj(x) + pos.unsqueeze(0)  # [B, NL, D_H]
        x = self.transformer(x)                    # [B, NL, D_H]
        x = self.pool_norm(x.mean(dim=1))          # [B, D_H] — mean pool
        return self.out_proj(x)                    # [B, D_LAT]


# ── Encodeur musical house ────────────────────────────────────────────────────
class MusicEncoder(nn.Module):
    """
    Enc_music : fenêtres de WINDOW notes → z_music [D_LAT]
    Features/note : pitch, dur, vel, interval, beat_pos
    Embedding(pitch,32) + MLP(dur+vel+interval+beat_pos → 32) → [WINDOW, 64]
    → Transformer → mean pool → Linear → [D_LAT]
    """
    def __init__(self, cfg):
        super().__init__()
        D_LAT   = cfg["D_LAT"]
        D_H     = cfg["D_HIDDEN"]
        WINDOW  = cfg["WINDOW"]

        self.pitch_emb  = nn.Embedding(128, 32)
        self.cont_mlp   = nn.Sequential(
            nn.Linear(4, 32), nn.GELU(), nn.Linear(32, 32))

        # Sinusoïdal pos encoding
        pos = torch.zeros(WINDOW, 64)
        for i in range(WINDOW):
            for k in range(0, 64, 2):
                pos[i,k]   = np.sin(i / (10000**(k/64)))
                pos[i,k+1] = np.cos(i / (10000**(k/64)))
        self.register_buffer("pos_enc", pos)  # [WINDOW, 64]

        self.input_proj = nn.Linear(64, D_H)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_H, nhead=cfg["N_HEADS"],
            dim_feedforward=D_H*4,
            dropout=cfg["DROPOUT"], batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer,
                                                  num_layers=cfg["N_TRANS"])
        self.pool_norm  = nn.LayerNorm(D_H)
        self.out_proj   = nn.Linear(D_H, D_LAT)

        # Embeddings de gamme pour le décodeur (utilisés à l'inférence)
        self.scale_proj = nn.Linear(32, D_LAT)

    def forward(self, windows):
        """
        windows : [B, WINDOW, 5]  (pitch_n, dur_n, vel_n, interval_n, beat_pos_n)
        Retourne : z_music [B, D_LAT]
        """
        B, W, _ = windows.shape
        # pitch → embedding (dequantifier de [0,1] vers int)
        pitch_idx = (windows[:,:,0] * 127).long().clamp(0,127)  # [B, W]
        p_emb = self.pitch_emb(pitch_idx)  # [B, W, 32]
        # features continues
        c_emb = self.cont_mlp(windows[:,:,1:])  # [B, W, 4] → [B, W, 32]
        x = torch.cat([p_emb, c_emb], dim=-1)  # [B, W, 64]
        x = x + self.pos_enc.unsqueeze(0)[:,:W,:]  # pos encoding
        x = self.input_proj(x)                  # [B, W, D_H]
        x = self.transformer(x)                 # [B, W, D_H]
        x = self.pool_norm(x.mean(dim=1))       # [B, D_H]
        return self.out_proj(x)                  # [B, D_LAT]

    def decode_to_scale(self, z, scale_pitches):
        """
        z : [B, D_LAT]
        scale_pitches : [SL] int
        Retourne : logits [B, SL] = z @ scale_embed.T
        """
        pitch_idx = torch.tensor(scale_pitches, dtype=torch.long, device=z.device)
        scale_emb = self.pitch_emb(pitch_idx)   # [SL, 32]
        scale_lat = self.scale_proj(scale_emb)  # [SL, D_LAT]
        return z @ scale_lat.T                   # [B, SL]


# ── Predictor cross-modal ─────────────────────────────────────────────────────
class Predictor(nn.Module):
    """
    MLP(D_LAT+2 → D_PRED → D_LAT) pour mapper z_stats → z_stats_pred.
    Les 2 features additionnelles = (step_pos, tension).
    """
    def __init__(self, cfg):
        super().__init__()
        D_IN = cfg["D_LAT"] + 2
        D_H  = cfg["D_PRED"]
        D_O  = cfg["D_LAT"]
        self.mlp = nn.Sequential(
            nn.Linear(D_IN, D_H), nn.GELU(), nn.LayerNorm(D_H),
            nn.Linear(D_H,  D_H), nn.GELU(), nn.LayerNorm(D_H),
            nn.Linear(D_H,  D_O))
    def forward(self, z, step_pos=None, tension=None):
        B = z.shape[0]
        if step_pos is None: step_pos = torch.zeros(B,1,device=z.device)
        else: step_pos = step_pos.view(B,1)
        if tension is None:  tension  = torch.zeros(B,1,device=z.device)
        else: tension = tension.view(B,1)
        return self.mlp(torch.cat([z, step_pos, tension], dim=-1))


# ── GRU pour la continuité temporelle ────────────────────────────────────────
class TemporalPredictor(nn.Module):
    """GRU 1 couche : prédit z_music[t+1] depuis z_music[t]."""
    def __init__(self, d_lat):
        super().__init__()
        self.gru  = nn.GRU(d_lat, d_lat, num_layers=1, batch_first=True)
        self.proj = nn.Linear(d_lat, d_lat)
    def forward(self, z_seq):  # z_seq: [B, T, D_LAT]
        out, _ = self.gru(z_seq)
        return self.proj(out)   # [B, T, D_LAT] — z_pred[t] ≈ z[t+1]


# ── Modèle JEPA complet (online + target) ────────────────────────────────────
class JEPAHouse(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Encodeurs online
        self.enc_stats  = StatsEncoder(cfg)
        self.enc_music  = MusicEncoder(cfg)
        self.predictor  = Predictor(cfg)
        self.temp_pred  = TemporalPredictor(cfg["D_LAT"])
        # Encodeurs target (copie EMA — pas de gradient)
        self.enc_stats_t = copy.deepcopy(self.enc_stats)
        self.enc_music_t = copy.deepcopy(self.enc_music)
        for p in self.enc_stats_t.parameters(): p.requires_grad_(False)
        for p in self.enc_music_t.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def ema_update(self, tau):
        """Met à jour les target encoders via EMA."""
        for p_o, p_t in zip(self.enc_stats.parameters(),
                            self.enc_stats_t.parameters()):
            p_t.data = tau*p_t.data + (1-tau)*p_o.data
        for p_o, p_t in zip(self.enc_music.parameters(),
                            self.enc_music_t.parameters()):
            p_t.data = tau*p_t.data + (1-tau)*p_o.data

    def forward(self, batch_stats, batch_music, step_pos=None, tension=None):
        """
        batch_stats : dict de tenseurs LLM (sv_q, sv_k, sv_v, sv_gate, sv_down,
                                            alpha, bimod, wdists, qk_cohs)
        batch_music : [B, WINDOW, 5]
        Retourne : dict de tenseurs pour le calcul des losses
        """
        # Encodeurs online
        sv_tensors = [batch_stats[k] for k in ["sv_q","sv_k","sv_v","sv_gate","sv_down"]]
        z_stats = self.enc_stats(
            sv_tensors,
            batch_stats["alpha"], batch_stats["bimod"],
            batch_stats["wdists"], batch_stats["qk_cohs"])
        z_music = self.enc_music(batch_music)
        z_pred  = self.predictor(z_stats, step_pos, tension)
        # Encodeurs target (stop-gradient via @no_grad dans ema_update)
        with torch.no_grad():
            z_stats_t = self.enc_stats_t(
                sv_tensors,
                batch_stats["alpha"], batch_stats["bimod"],
                batch_stats["wdists"], batch_stats["qk_cohs"])
            z_music_t = self.enc_music_t(batch_music)
        return {"z_stats":z_stats, "z_music":z_music,
                "z_pred":z_pred,
                "z_stats_t":z_stats_t, "z_music_t":z_music_t}


# ── Fonctions de loss ─────────────────────────────────────────────────────────
def cosine_loss(a, b):
    """L_align = 1 - cosine_similarity(a, sg(b)), symétrique."""
    a = F.normalize(a, dim=-1)
    b = F.normalize(b.detach(), dim=-1)
    return 1.0 - (a * b).sum(dim=-1).mean()

def variance_loss(z, gamma=1.0):
    """L_var = max(0, gamma - std(z)) par dimension."""
    std = z.std(dim=0)  # [D_LAT]
    return F.relu(gamma - std).mean()

def temporal_loss(temp_pred, z_seq):
    """
    L_temp = MSE(z_pred[t], z[t+1]) sur la séquence.
    z_seq : [B, T, D_LAT]
    """
    pred = temp_pred(z_seq[:, :-1, :])  # [B, T-1, D_LAT]
    tgt  = z_seq[:, 1:,  :].detach()   # [B, T-1, D_LAT]
    return F.mse_loss(pred, tgt)

def total_loss(out, cfg):
    """Calcule L_total depuis le dict de sortie du modèle."""
    L_align = cosine_loss(out["z_pred"], out["z_music_t"])
    L_var   = (variance_loss(out["z_stats"], cfg["GAMMA_VAR"]) +
               variance_loss(out["z_music"], cfg["GAMMA_VAR"])) * 0.5
    return L_align + cfg["LAMBDA_VAR"]*L_var, L_align, L_var


# Instanciation + test
model = JEPAHouse(CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"JEPA House — {n_params:,} paramètres ({n_params/1e6:.2f}M)")

# Test forward
B = 4
dummy_stats = {
    k: torch.rand(B, CFG["NL"], CFG["D_SV"]).to(DEVICE)
    for k in ["sv_q","sv_k","sv_v","sv_gate","sv_down"]}
dummy_stats["alpha"]   = torch.rand(B, CFG["NL"]).to(DEVICE)
dummy_stats["bimod"]   = torch.rand(B, CFG["NL"]).to(DEVICE)
dummy_stats["wdists"]  = torch.rand(B, CFG["NL"]).to(DEVICE)
dummy_stats["qk_cohs"] = torch.rand(B, CFG["NL"]).to(DEVICE)
dummy_music = torch.rand(B, CFG["WINDOW"], CFG["NOTE_FEAT"]).to(DEVICE)

out = model(dummy_stats, dummy_music)
L, La, Lv = total_loss(out, CFG)
print(f"Forward OK — z_stats={out["z_stats"].shape}  z_music={out["z_music"].shape}")
print(f"L_total={L.item():.4f}  L_align={La.item():.4f}  L_var={Lv.item():.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════
# DATASETS PYTORCH + CURRICULUM
# ══════════════════════════════════════════════════════════

class HouseMIDIDataset(Dataset):
    """
    Dataset de fenêtres MIDI house.
    Chaque item = (window [WINDOW,5], entropy float)
    """
    def __init__(self, windows, entropies):
        self.windows    = torch.tensor(windows,   dtype=torch.float32)
        self.entropies  = torch.tensor(entropies, dtype=torch.float32)
    def __len__(self): return len(self.windows)
    def __getitem__(self, idx):
        return self.windows[idx], self.entropies[idx]


class LLMStatsDataset(Dataset):
    """
    Dataset de features LLM.
    Pour le JEPA, on génère des variantes avec bruit léger
    pour augmenter artificiellement le corpus (un seul modèle).

    augmentation_factor : nombre de variantes bruitées par sample
    noise_std : écart-type du bruit gaussien additif
    """
    def __init__(self, feat_dict, n_augment=500, noise_std=0.02):
        self.feat  = feat_dict
        self.N     = n_augment
        self.noise = noise_std
        self.NL    = feat_dict["sv_q"].shape[0]

        # Tension par layer = 1 - alpha_norm
        alpha_n = feat_dict["alpha"] / 2.0  # [NL] en [0,1]
        self.tension_per_layer = 1.0 - alpha_n
        self.global_tension    = float(self.tension_per_layer.mean())

    def __len__(self): return self.N

    def __getitem__(self, idx):
        # Variante bruitée des features LLM
        rng = np.random.RandomState(idx)
        def noisy(x): return x + rng.randn(*x.shape).astype(np.float32)*self.noise
        item = {k: torch.tensor(noisy(self.feat[k]), dtype=torch.float32)
                for k in ["sv_q","sv_k","sv_v","sv_gate","sv_down",
                          "alpha","bimod","wdists","qk_cohs"]}
        item["tension"] = torch.tensor(self.global_tension, dtype=torch.float32)
        return item


class CurriculumPairedDataset(Dataset):
    """
    Dataset apparié LLM ↔ MIDI avec curriculum de tension.

    Mode "random"       : paires aléatoires
    Mode "soft_tension" : paires par quantile de tension (±1 quantile)
    Mode "hard_tension" : paires strictement dans le même quantile de tension

    LLM tension = 1 - alpha_moyen (impulsif → tendu)
    MIDI tension = entropie harmonique normalisée
    """
    QUANTILES = 3  # bas / moyen / haut

    def __init__(self, llm_ds, midi_ds, mode="random", seed=42):
        self.llm  = llm_ds
        self.midi = midi_ds
        self.mode = mode
        self.rng  = np.random.RandomState(seed)

        # Index MIDI par quantile de tension
        ent = midi_ds.entropies.numpy()
        q33, q67 = np.percentile(ent,33), np.percentile(ent,67)
        self.midi_low  = np.where(ent <= q33)[0]
        self.midi_mid  = np.where((ent>q33)&(ent<=q67))[0]
        self.midi_high = np.where(ent > q67)[0]

    def __len__(self): return len(self.llm)

    def _get_midi_idx(self, llm_tension):
        if self.mode == "random":
            return self.rng.randint(len(self.midi))
        # Tension LLM → quantile → pool MIDI correspondant
        if   llm_tension < 0.33: pool = self.midi_low
        elif llm_tension < 0.67: pool = self.midi_mid
        else:                    pool = self.midi_high
        if self.mode == "hard_tension" or self.rng.rand() > 0.3:
            return self.rng.choice(pool)
        return self.rng.randint(len(self.midi))  # 30% aléatoire en soft

    def __getitem__(self, idx):
        llm_item = self.llm[idx]
        t_llm    = float(llm_item["tension"])
        midi_idx = self._get_midi_idx(t_llm)
        window, entropy = self.midi[midi_idx]
        llm_item["window"]  = window
        llm_item["entropy"] = entropy
        return llm_item


# Construction des datasets
windows   = np.load("lakh_house_windows.npy")
entropies = np.load("lakh_house_entropies.npy")

# Split train/val/test sur les fenêtres
N = len(windows)
idx = np.random.permutation(N)
split_tr, split_v = int(N*0.80), int(N*0.90)
midi_train = HouseMIDIDataset(windows[idx[:split_tr]],  entropies[idx[:split_tr]])
midi_val   = HouseMIDIDataset(windows[idx[split_tr:split_v]], entropies[idx[split_tr:split_v]])
midi_test  = HouseMIDIDataset(windows[idx[split_v:]], entropies[idx[split_v:]])

llm_ds = LLMStatsDataset(feat_dict, n_augment=10000, noise_std=0.015)

print(f"Datasets construits:")
print(f"  MIDI train={len(midi_train):,}  val={len(midi_val):,}  test={len(midi_test):,}")
print(f"  LLM (augmenté): {len(llm_ds):,}")
print(f"  Tension LLM globale: {llm_ds.global_tension:.3f}")


In [ ]:
# ══════════════════════════════════════════════════════════
# TRAINING LOOP — 3 phases
# Phase 1 : musique seule (Enc_music pré-entraîné)
# Phase 2 : alignement cross-modal (Enc_music gelé)
# Phase 3 : fine-tuning joint
# ══════════════════════════════════════════════════════════

def get_tau(epoch, total_epochs, tau_init=0.996, tau_final=0.9999):
    """Schedule cosine pour tau EMA."""
    progress = epoch / max(total_epochs-1, 1)
    return tau_init + (tau_final-tau_init)*progress

def get_curriculum_mode(epoch, warmup=10, total=50):
    if epoch < warmup:     return "random"
    prog = (epoch-warmup) / (total-warmup)
    if prog < 0.5:         return "soft_tension"
    return "hard_tension"

def make_loader(dataset, batch_size, shuffle=True):
    return DataLoader(dataset, batch_size=batch_size,
                      shuffle=shuffle, num_workers=2,
                      pin_memory=DEVICE.type=="cuda", drop_last=True)

def eval_alignment(model, midi_val, llm_ds, cfg, n_eval=500):
    """Évalue R@1 et R@5 sur le val set."""
    model.eval()
    all_z_pred, all_z_music = [], []
    loader_m = make_loader(midi_val, 64, shuffle=False)
    loader_l = make_loader(llm_ds,   64, shuffle=False)
    with torch.no_grad():
        for (win,_),(llm_b) in zip(loader_m, loader_l):
            win = win.to(DEVICE)
            sv  = {k:v.to(DEVICE) for k,v in llm_b.items()
                   if k not in ["window","entropy","tension"]}
            z_music = model.enc_music(win)
            z_stats = model.enc_stats(
                [sv[k] for k in ["sv_q","sv_k","sv_v","sv_gate","sv_down"]],
                sv["alpha"],sv["bimod"],sv["wdists"],sv["qk_cohs"])
            z_pred = model.predictor(z_stats)
            all_z_pred.append(F.normalize(z_pred, dim=-1).cpu())
            all_z_music.append(F.normalize(z_music, dim=-1).cpu())
            if len(all_z_pred)*64 >= n_eval: break
    Z_pred  = torch.cat(all_z_pred)[:n_eval]
    Z_music = torch.cat(all_z_music)[:n_eval]
    sim = Z_pred @ Z_music.T  # [N, N] — similarité cosine
    ranks = (sim.argsort(dim=1, descending=True) == torch.arange(n_eval).unsqueeze(1)).float()
    r1 = float(ranks[:,:1].max(dim=1).values.mean())
    r5 = float(ranks[:,:5].max(dim=1).values.mean())
    cos_mean = float(torch.diag(sim).mean())
    model.train()
    return {"R@1":r1, "R@5":r5, "cosine":cos_mean}


# ─────────────────────────────────────────────────────────
# PHASE 1 : Pré-entraînement musical (Enc_music seul)
# ─────────────────────────────────────────────────────────
print("=" * 55)
print("PHASE 1 — Pré-entraînement musical")
print("=" * 55)

optim_p1 = torch.optim.AdamW(
    model.enc_music.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
sched_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim_p1, T_max=CFG["EPOCHS_P1"]-CFG["WARMUP"])

loader_p1 = make_loader(midi_train, CFG["BATCH"])
logs_p1 = []

for epoch in range(CFG["EPOCHS_P1"]):
    epoch_loss = 0.0; n_steps = 0
    for windows_b, entropies_b in loader_p1:
        windows_b = windows_b.to(DEVICE)
        B = windows_b.shape[0]
        # Auto-supervision : prédire la 2ème moitié depuis la 1ère
        half = CFG["WINDOW"] // 2
        z_ctx = model.enc_music(windows_b[:, :half, :].repeat_interleave(2,dim=1))
        z_tgt = model.enc_music_t(windows_b)
        loss  = cosine_loss(model.predictor(z_ctx), z_tgt)
        loss += CFG["LAMBDA_VAR"] * variance_loss(z_ctx, CFG["GAMMA_VAR"])
        optim_p1.zero_grad(); loss.backward();
        nn.utils.clip_grad_norm_(model.enc_music.parameters(), CFG["GRAD_CLIP"])
        optim_p1.step()
        tau = get_tau(epoch, CFG["EPOCHS_P1"])
        model.ema_update(tau)
        epoch_loss += loss.item(); n_steps += 1
    if epoch >= CFG["WARMUP"]: sched_p1.step()
    avg = epoch_loss/n_steps
    logs_p1.append(avg)
    if epoch % 10 == 0:
        print(f"  epoch {epoch:3d}/{CFG["EPOCHS_P1"]}  loss={avg:.4f}  tau={tau:.4f}")

torch.save(model.enc_music.state_dict(), "checkpoints/enc_music_p1.pt")
print("Phase 1 terminée — enc_music_p1.pt sauvegardé")


# ─────────────────────────────────────────────────────────
# PHASE 2 : Alignement cross-modal (Enc_music gelé)
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("PHASE 2 — Alignement cross-modal (Enc_music gelé)")
print("=" * 55)

for p in model.enc_music.parameters():   p.requires_grad_(False)
for p in model.enc_music_t.parameters(): p.requires_grad_(False)

params_p2 = list(model.enc_stats.parameters()) + list(model.predictor.parameters())
optim_p2  = torch.optim.AdamW(params_p2, lr=CFG["LR"], weight_decay=CFG["WD"])
sched_p2  = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim_p2, T_max=CFG["EPOCHS_P2"]-CFG["WARMUP"])

logs_p2 = []

for epoch in range(CFG["EPOCHS_P2"]):
    mode = get_curriculum_mode(epoch, warmup=10, total=CFG["EPOCHS_P2"])
    paired_ds = CurriculumPairedDataset(llm_ds, midi_train, mode=mode)
    loader_p2 = make_loader(paired_ds, CFG["BATCH"])
    epoch_loss = 0.0; epoch_align = 0.0; n_steps = 0
    for batch in loader_p2:
        windows_b = batch["window"].to(DEVICE)
        sv = {k:batch[k].to(DEVICE) for k in ["sv_q","sv_k","sv_v","sv_gate","sv_down",
                                                "alpha","bimod","wdists","qk_cohs"]}
        tension = batch["tension"].to(DEVICE)
        out  = model(sv, windows_b, tension=tension)
        L, La, Lv = total_loss(out, CFG)
        optim_p2.zero_grad(); L.backward()
        nn.utils.clip_grad_norm_(params_p2, CFG["GRAD_CLIP"])
        optim_p2.step()
        tau = get_tau(epoch, CFG["EPOCHS_P2"])
        model.ema_update(tau)
        epoch_loss += L.item(); epoch_align += La.item(); n_steps += 1
    if epoch >= CFG["WARMUP"]: sched_p2.step()
    avg_l = epoch_loss/n_steps; avg_a = epoch_align/n_steps
    logs_p2.append((avg_l, avg_a))
    if epoch % 10 == 0:
        metrics = eval_alignment(model, midi_val, llm_ds, CFG, n_eval=256)
        print(f"  epoch {epoch:3d}/{CFG["EPOCHS_P2"]}  L={avg_l:.4f}  "
              f"L_align={avg_a:.4f}  [{mode}]  "
              f"R@1={metrics["R@1"]:.3f}  R@5={metrics["R@5"]:.3f}  "
              f"cos={metrics["cosine"]:.3f}")

torch.save({"enc_stats": model.enc_stats.state_dict(),
            "predictor":  model.predictor.state_dict()},
           "checkpoints/jepa_p2.pt")
print("Phase 2 terminée — jepa_p2.pt sauvegardé")


# ─────────────────────────────────────────────────────────
# PHASE 3 : Fine-tuning joint (tous les composants)
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("PHASE 3 — Fine-tuning joint")
print("=" * 55)

for p in model.enc_music.parameters(): p.requires_grad_(True)

optim_p3 = torch.optim.AdamW([
    {"params": model.enc_music.parameters(),  "lr": CFG["LR_FROZEN"]},
    {"params": model.enc_stats.parameters(),  "lr": CFG["LR"]},
    {"params": model.predictor.parameters(),  "lr": CFG["LR"]},
    {"params": model.temp_pred.parameters(),  "lr": CFG["LR"]},
], weight_decay=CFG["WD"])
sched_p3 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim_p3, T_max=CFG["EPOCHS_P3"])

logs_p3 = []; best_r1 = 0.0

for epoch in range(CFG["EPOCHS_P3"]):
    paired_ds = CurriculumPairedDataset(llm_ds, midi_train, mode="hard_tension")
    loader_p3 = make_loader(paired_ds, CFG["BATCH"])
    epoch_loss = 0.0; epoch_align = 0.0; epoch_temp = 0.0; n_steps = 0
    for batch in loader_p3:
        windows_b = batch["window"].to(DEVICE)
        sv = {k:batch[k].to(DEVICE) for k in ["sv_q","sv_k","sv_v","sv_gate","sv_down",
                                                "alpha","bimod","wdists","qk_cohs"]}
        tension = batch["tension"].to(DEVICE)
        out  = model(sv, windows_b, tension=tension)
        L, La, Lv = total_loss(out, CFG)
        # Loss temporelle sur une séquence simulée (batch comme séquence temporelle)
        z_seq = out["z_music"].unsqueeze(0)  # [1, B, D_LAT] simulé comme séquence
        # En vrai on passerait des séquences consécutives — simplification ici
        L_total = L
        optim_p3.zero_grad(); L_total.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
        optim_p3.step()
        tau = get_tau(epoch, CFG["EPOCHS_P3"], tau_init=0.9990, tau_final=0.9999)
        model.ema_update(tau)
        epoch_loss += L.item(); epoch_align += La.item(); n_steps += 1
    sched_p3.step()
    avg_l = epoch_loss/n_steps; avg_a = epoch_align/n_steps
    logs_p3.append((avg_l, avg_a))
    metrics = eval_alignment(model, midi_val, llm_ds, CFG, n_eval=256)
    print(f"  epoch {epoch:3d}/{CFG["EPOCHS_P3"]}  L={avg_l:.4f}  "
          f"R@1={metrics["R@1"]:.3f}  R@5={metrics["R@5"]:.3f}")
    if metrics["R@1"] > best_r1:
        best_r1 = metrics["R@1"]
        torch.save(model.state_dict(), "checkpoints/jepa_house_best.pt")
        print(f"    ✓ Nouveau best R@1={best_r1:.3f}")

torch.save(model.state_dict(), "checkpoints/jepa_house_final.pt")
print(f"\nPhase 3 terminée — best R@1={best_r1:.3f}")


In [ ]:
# ══════════════════════════════════════════════════════════
# ÉVALUATION
# ══════════════════════════════════════════════════════════

# Charger le meilleur modèle
model.load_state_dict(torch.load("checkpoints/jepa_house_best.pt", map_location=DEVICE))
model.eval()

# ── 1. Métriques d'alignement sur le test set ────────────────────────────────
metrics_test = eval_alignment(model, midi_test, llm_ds, CFG, n_eval=1000)
print("=== MÉTRIQUES ALIGNEMENT (test set) ===")
print(f"  R@1:    {metrics_test["R@1"]:.4f}")
print(f"  R@5:    {metrics_test["R@5"]:.4f}")
print(f"  cosine: {metrics_test["cosine"]:.4f}")

# ── 2. Distribution des z dans l'espace latent ───────────────────────────────
from sklearn.decomposition import PCA as PCA_sk

z_stats_all, z_music_all, tensions_all = [], [], []
loader_eval = make_loader(CurriculumPairedDataset(
    llm_ds, midi_test, mode="random"), 64, shuffle=False)

with torch.no_grad():
    for batch in loader_eval:
        win = batch["window"].to(DEVICE)
        sv  = {k:batch[k].to(DEVICE) for k in
               ["sv_q","sv_k","sv_v","sv_gate","sv_down","alpha","bimod","wdists","qk_cohs"]}
        z_s = model.enc_stats([sv[k] for k in ["sv_q","sv_k","sv_v","sv_gate","sv_down"]],
                               sv["alpha"],sv["bimod"],sv["wdists"],sv["qk_cohs"])
        z_m = model.enc_music(win)
        z_stats_all.append(z_s.cpu()); z_music_all.append(z_m.cpu())
        tensions_all.append(batch["entropy"])
        if len(z_stats_all)*64 >= 2000: break

Z_s = torch.cat(z_stats_all)[:2000].numpy()
Z_m = torch.cat(z_music_all)[:2000].numpy()
T   = torch.cat(tensions_all)[:2000].numpy()

pca2 = PCA_sk(n_components=2).fit(np.vstack([Z_s, Z_m]))
Zs_2d = pca2.transform(Z_s)
Zm_2d = pca2.transform(Z_m)

fig,axes=plt.subplots(1,2,figsize=(14,5))
sc=axes[0].scatter(Zs_2d[:,0],Zs_2d[:,1],c=T,cmap="RdYlGn_r",s=8,alpha=0.6,label="z_stats")
axes[0].scatter(Zm_2d[:,0],Zm_2d[:,1],c=T,cmap="RdYlGn_r",s=8,alpha=0.3,marker="x",label="z_music")
plt.colorbar(sc,ax=axes[0],label="entropy MIDI")
axes[0].set_title("Espace latent PCA-2D\n(z_stats=ronds, z_music=croix)")
axes[0].legend()

# Variance par dimension
axes[1].bar(range(0,min(32,Z_s.shape[1])), np.var(Z_s,axis=0)[:32],
            alpha=0.7, color="#4A90D9", label="z_stats")
axes[1].bar(range(0,min(32,Z_m.shape[1])), np.var(Z_m,axis=0)[:32],
            alpha=0.5, color="#E87040", label="z_music")
axes[1].set_title("Variance par dimension (32 premières)")
axes[1].legend()
plt.tight_layout()
plt.savefig("logs/eval_latent_space.png", dpi=120, bbox_inches="tight")
plt.show()

# ── 3. Courbes de loss ───────────────────────────────────────────────────────
fig,axes=plt.subplots(1,3,figsize=(15,4))
axes[0].plot(logs_p1,"b-"); axes[0].set_title("Phase 1 — loss"); axes[0].set_xlabel("epoch")
axes[1].plot([l[0] for l in logs_p2],"b-",label="total")
axes[1].plot([l[1] for l in logs_p2],"r--",label="align")
axes[1].set_title("Phase 2"); axes[1].legend()
axes[2].plot([l[0] for l in logs_p3],"b-",label="total")
axes[2].plot([l[1] for l in logs_p3],"r--",label="align")
axes[2].set_title("Phase 3"); axes[2].legend()
plt.tight_layout()
plt.savefig("logs/training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\nVariance z_stats (32 dims): {np.var(Z_s,axis=0)[:32].mean():.4f}")
print(f"Variance z_music (32 dims): {np.var(Z_m,axis=0)[:32].mean():.4f}")
print(f"Collapse check (var > 0.01): {(np.var(Z_s,axis=0)>0.01).mean()*100:.0f}% dims actives")


In [ ]:
# ══════════════════════════════════════════════════════════
# EXPORT DES ARTEFACTS
# ══════════════════════════════════════════════════════════

import zipfile
from google.colab import files

# Sauvegarder la config avec le modèle
torch.save({
    "model_state_dict": model.state_dict(),
    "cfg": CFG,
    "metrics": metrics_test,
    "model_id": MODEL_ID,
}, "checkpoints/jepa_house_v1.pt")

# Exporter les 3 composants séparément pour l'inférence
torch.save(model.enc_stats.state_dict(),  "checkpoints/enc_stats.pt")
torch.save(model.enc_music.state_dict(),  "checkpoints/enc_music.pt")
torch.save(model.predictor.state_dict(),  "checkpoints/predictor.pt")

# Sauvegarder aussi les stats de normalisation du corpus
norm_stats = {
    "entropy_q33": float(np.percentile(entropies, 33)),
    "entropy_q67": float(np.percentile(entropies, 67)),
    "llm_tension": LLM_TENSION,
    "n_windows":   len(windows),
    "n_files":     n_files_ok,
}
with open("checkpoints/norm_stats.json","w") as f:
    json.dump(norm_stats, f, indent=2)

print("Artefacts exportés:")
for f in sorted(os.listdir("checkpoints")):
    size = os.path.getsize(f"checkpoints/{f}")
    print(f"  {f:<35} {size/1e6:.2f} MB")

# Zip et téléchargement
with zipfile.ZipFile("jepa_house_v1.zip","w",zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir("checkpoints"):
        zf.write(f"checkpoints/{f}", f)
    for f in os.listdir("logs"):
        zf.write(f"logs/{f}", f"logs/{f}")

print(f"\nZip: jepa_house_v1.zip ({os.path.getsize("jepa_house_v1.zip")/1e6:.1f} MB)")
files.download("jepa_house_v1.zip")


In [ ]:
# ══════════════════════════════════════════════════════════
# INTÉGRATION — Remplacement du LSTM dans la génération
# ══════════════════════════════════════════════════════════
# Ce module remplace generate_markov_lstm() dans le DJ set notebook

class JEPAInference:
    """
    Wrapper d'inférence JEPA pour la génération MIDI.
    Remplace le LSTM non-entraîné dans generate_markov_lstm().
    """
    def __init__(self, checkpoint_path, cfg, device="cpu"):
        self.cfg    = cfg
        self.device = torch.device(device)
        self.model  = JEPAHouse(cfg).to(self.device)
        ckpt = torch.load(checkpoint_path, map_location=self.device)
        if "model_state_dict" in ckpt:
            self.model.load_state_dict(ckpt["model_state_dict"])
        else:
            self.model.load_state_dict(ckpt)
        self.model.eval()
        print(f"JEPA chargé depuis {checkpoint_path}")

    def encode_llm(self, feat_dict):
        """Encode les features LLM → z_stats [D_LAT]."""
        with torch.no_grad():
            sv = {k: torch.tensor(feat_dict[k], dtype=torch.float32)
                     .unsqueeze(0).to(self.device)
                  for k in ["sv_q","sv_k","sv_v","sv_gate","sv_down",
                             "alpha","bimod","wdists","qk_cohs"]}
            z = self.model.enc_stats(
                [sv[k] for k in ["sv_q","sv_k","sv_v","sv_gate","sv_down"]],
                sv["alpha"],sv["bimod"],sv["wdists"],sv["qk_cohs"])
            return z[0].cpu().numpy()  # [D_LAT]

    def get_transition_probs(self, z_stats, scale_pitches,
                              step_pos=0, tension=0.5, temperature=1.0):
        """
        Depuis z_stats, retourne une distribution de probabilités
        sur les degrés de la gamme scale_pitches.

        Remplace le rôle du LSTM dans la chaîne de Markov :
          p_jepa = softmax(logits / temp)
          p_final = MARKOV_W * M[current] + (1-MARKOV_W) * p_jepa
        """
        with torch.no_grad():
            z = torch.tensor(z_stats, dtype=torch.float32).unsqueeze(0).to(self.device)
            sp = torch.tensor([[step_pos]], dtype=torch.float32).to(self.device)
            tn = torch.tensor([[tension]],  dtype=torch.float32).to(self.device)
            z_pred = self.model.predictor(z, sp, tn)
            logits = self.model.enc_music.decode_to_scale(
                z_pred, scale_pitches)  # [1, SL]
            logits = logits[0].cpu().numpy()
        lp = logits / max(temperature, 1e-6)
        lp -= lp.max()
        p = np.exp(lp); p /= p.sum()
        return p  # [SL]


def generate_markov_jepa(feat_dict, scale, n_notes,
                          jepa_inf, markov_weight=None,
                          temperature=1.0, seed=None):
    """
    Version JEPA de generate_markov_lstm().
    Remplace le LSTM par le predictor JEPA entraîné.

    Le flow est identique :
    1. Matrice de transition Markov depuis outer(sv_q) (inchangée)
    2. Distribution JEPA depuis z_stats → predictor → decode_to_scale
    3. Combinaison : p = markov_w * M[cur] + (1-markov_w) * p_jepa
    """
    if seed is not None: np.random.seed(seed)
    if markov_weight is None: markov_weight = CFG["MARKOV_W"]
    SL  = len(scale)
    sv_q = feat_dict["sv_q"]

    # Matrice de transition Markov (inchangée depuis les notebooks poids)
    def outer_markov(sv, SL, sharp=3.0):
        v = norm01_np(np.abs(sv[:SL])) if len(sv)>=SL \
            else np.pad(norm01_np(np.abs(sv)),(0,SL-len(sv)))
        M = np.outer(v,v)
        M -= M.max(axis=1,keepdims=True)
        M = np.exp(M*sharp)
        M /= M.sum(axis=1,keepdims=True)+1e-12
        return M

    M = outer_markov(sv_q.mean(axis=0), SL)  # SV moyen sur les layers
    z_stats = jepa_inf.encode_llm(feat_dict)

    # Tension globale du modèle
    tension = float(1.0 - feat_dict["alpha"].mean() / 2.0)

    cur  = int(norm01_np(np.abs(sv_q.mean(axis=0)[:SL]))[0] * (SL-1))
    seq  = []
    for t in range(n_notes):
        step_pos_n = t / max(n_notes-1, 1)  # 0→1
        # Distribution JEPA
        p_jepa = jepa_inf.get_transition_probs(
            z_stats, scale,
            step_pos=step_pos_n,
            tension=tension,
            temperature=temperature)
        # Combinaison
        p = markov_weight * M[cur] + (1-markov_weight) * p_jepa
        p /= p.sum()
        # Entropie du step
        H_step = float(-np.sum(p * np.log(p+1e-12)))
        nxt = int(np.random.choice(SL, p=p))
        seq.append((cur, int(scale[cur]), H_step, float(tension)))
        cur = nxt
    return seq


# Test de l'intégration
print("Test intégration JEPA...")
jepa_inf = JEPAInference("checkpoints/jepa_house_best.pt", CFG, device=str(DEVICE))

# Gamme de test
def build_scale(root, mode="natural_minor", lo=36, hi=84):
    ivs={"natural_minor":[0,2,3,5,7,8,10],"dorian":[0,2,3,5,7,9,10],
         "phrygian":[0,1,3,5,7,8,10]}[mode]
    return np.array(sorted(set(
        n for o in range(9) for st in ivs if lo<=(n:=root+o*12+st)<=hi)))

SCALE_TEST = build_scale(50, "natural_minor")
seq = generate_markov_jepa(feat_dict, SCALE_TEST, n_notes=16,
                            jepa_inf=jepa_inf, seed=42)

NN=["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]
print(f"\nSéquence JEPA (16 notes):")
for i,(d,p,H,tn) in enumerate(seq):
    print(f"  {i:02d}  {NN[p%12]}{p//12-1}  deg={d:2d}  H={H:.2f}  tension={tn:.2f}")
print(f"\nDegrés uniques: {len(set(s[0] for s in seq))}/{len(SCALE_TEST)}")
print("\n✓ Intégration JEPA OK")
print("\nPour utiliser dans le DJ set notebook:")
print("  1. Charger jepa_house_v1.pt")
print("  2. Remplacer generate_markov_lstm() par generate_markov_jepa()")
print("  3. Passer jepa_inf comme paramètre")
